In [2]:
import wikipedia
import networkx as nx
from collections import deque

# ==========================
# CONFIGURAÇÕES
# ==========================

SEEDS = [
    "Cristiano Ronaldo",
    "Mount Everest", 
    "Python (programming language)",
    "French Revolution",
    "Albert Einstein"
]

MAX_DEPTH = 2
MAX_LINKS_PER_PAGE = 30  # Pega os top 15 links mais relevantes
MIN_CONTENT_LENGTH = 500 # Ignora páginas muito curtas (stubs)

# Palavras/Prefixos para ignorar
STOPS = {
    "Portal:", "Category:", "Help:", "Draft:", "Template:", 
    "Module:", "File:", "Image:", "Wikipedia:", "Talk:", 
    "User:", "MediaWiki:", "Special:", "Bibcode", "Doi", "Isbn", "Issn"
}

# Palavras genéricas para filtrar títulos
GENERAL_WORDS = {
    "history", "geography", "culture", "economy",
    "politics", "science", "sport", "language",
    "country", "region", "list", "outline", "bibliography",
    "timeline", "glossary", "index"
}

g = nx.DiGraph()
edges_temp = [] 

# ==========================
# HEURÍSTICA 
# ==========================

def get_best_links_fast(page_obj, max_links):
    """
    Analisa os links presentes na página e pontua baseando-se
    no conteúdo JÁ BAIXADO (frequência no texto e posição).
    """
    scored_links = []
    
    # Prepara textos para busca rápida (cache local)
    try:
        content_lower = page_obj.content.lower()
        summary_lower = page_obj.summary.lower()
        title_parts = set(page_obj.title.lower().split())
    except:
        return []

    for link in page_obj.links:
        link_lower = link.lower()
        
        # --- FILTROS ---
        # 1. Namespaces administrativos
        if any(link.startswith(s) for s in STOPS):
            continue
            
        # 2. Listas e Genéricos
        if link_lower.startswith("list of"):
            continue
        if any(w in link_lower for w in GENERAL_WORDS):
            continue
            
        # 3. Anos ou Números isolados (ex: "2023", "1990")
        if link.isdigit(): 
            continue

        # --- PONTUAÇÃO ---
        score = 0
        
        # A: Frequência no texto principal (Relevância bruta)
        # Limitamos a contagem a 50 para evitar distorções em textos gigantes
        count = content_lower.count(link_lower)
        score += min(count, 50)
        
        # B: Está no Resumo? (Alta relevância conceitual)
        if link_lower in summary_lower:
            score += 20
            
        # C: Compartilha palavras com o título da página pai? (Contexto)
        link_parts = set(link_lower.split())
        if not title_parts.isdisjoint(link_parts):
            score += 10
            
        # Penalidade: Links muito longos (geralmente frases ou lixo)
        if len(link_parts) > 5:
            score -= 10

        # Só adiciona se tiver pontuação positiva
        if score > 0:
            scored_links.append((score, link))

    # Ordena decrescente pelo score
    scored_links.sort(reverse=True)
    
    # Retorna apenas os nomes dos top links
    return [link for _, link in scored_links[:max_links]]

# ==========================
# BFS (BUSCA EM LARGURA)
# ==========================

def explore_seed(start_page):
    queue = deque()
    queue.append((0, start_page))

    visited = set()
    failed = set()

    print(f"\n{'='*40}")
    print(f"SEED: {start_page}")
    print(f"{'='*40}")

    while queue:
        layer, page_title = queue.popleft()

        # Limite de profundidade
        if layer > MAX_DEPTH:
            continue

        # Evitar re-visitar no mesmo loop
        if page_title in visited:
            continue
        visited.add(page_title)

        prefix = "  " * layer
        print(f"{prefix}➤ [L{layer}] Acessando: {page_title}...", end=" ", flush=True)

        try:
            # O gargalo é AQUI. Fazemos apenas 1 request por nó.
            wiki_obj = wikipedia.page(page_title, auto_suggest=False)
            print("OK.")
        
        except wikipedia.exceptions.DisambiguationError as e:
            print("⚠ Desambiguação (Pulado)")
            # Opcional: pegar a primeira opção da desambiguação
            # if layer < MAX_DEPTH: queue.append((layer, e.options[0]))
            continue
        except wikipedia.exceptions.PageError:
            print("❌ Não encontrado")
            failed.add(page_title)
            continue
        except Exception as e:
            print(f"❌ Erro: {e}")
            failed.add(page_title)
            continue

        # Filtro de conteúdo mínimo
        if len(wiki_obj.content) < MIN_CONTENT_LENGTH:
            print(f"{prefix}   ↳ Ignorado (Texto muito curto)")
            continue

        # Se atingiu o limite de profundidade, não precisamos buscar links,
        # apenas adicionamos o nó ao grafo (processamento acima) e paramos.
        if layer == MAX_DEPTH:
            continue

        # --- APLICA A NOVA HEURÍSTICA ---
        # Passamos o objeto já baixado (wiki_obj)
        best_links = get_best_links_fast(wiki_obj, MAX_LINKS_PER_PAGE)

        for link in best_links:
            # Normalização simples
            link_clean = link.strip().replace("_", " ") # Títulos da wiki geralmente não têm _
            
            # Adiciona à lista de arestas
            edges_temp.append((page_title, link_clean))

            if link_clean not in visited and link_clean not in failed:
                queue.append((layer + 1, link_clean))

# ==========================
# EXECUÇÃO PRINCIPAL
# ==========================

for seed in SEEDS:
    explore_seed(seed)

# Construir grafo
g.add_edges_from(edges_temp)

print("\nProcessing Graph Cleanup...")

# 1. Remover Self-Loops
g.remove_edges_from(nx.selfloop_edges(g))

# 2. Unificar Plurais (Simples)
nodes_list = list(g.nodes())
mapping = {}
for node in nodes_list:
    plural = node + "s"
    if plural in g:
        mapping[plural] = node # Mapeia plural -> singular

if mapping:
    print(f"Unificando {len(mapping)} plurais...")
    g = nx.relabel_nodes(g, mapping) # Relabel é mais seguro que contract para strings

# 3. Unificar Case (Maiúscula/Minúscula)
# Wikipedia é Case Sensitive na primeira letra, mas vamos padronizar
mapping_case = {}
nodes_lower = {n.lower(): n for n in g.nodes()} # mapa lower -> original
for node in g.nodes():
    if node.lower() in nodes_lower and node != nodes_lower[node.lower()]:
        # Se existe versão alternativa, mapear para a que já existe
        mapping_case[node] = nodes_lower[node.lower()]

if mapping_case:
    g = nx.relabel_nodes(g, mapping_case)

# ==========================
# SALVAR E ESTATÍSTICAS
# ==========================

filename = "graph_wikipedia_opt.graphml"
nx.write_graphml(g, filename)

print(f"\n{'='*30}")
print(" RESULTADOS FINAIS")
print(f"{'='*30}")
print(f"Total de Nós:     {g.number_of_nodes()}")
print(f"Total de Arestas: {g.number_of_edges()}")
print(f"Arquivo Salvo:    {filename}")

# Bonus: Top 5 Centralidade de Grau (Quem recebeu mais links?)
try:
    in_degree = dict(g.in_degree())
    top_nodes = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop 5 Páginas mais citadas no grafo:")
    for page, degree in top_nodes:
        print(f" - {page}: {degree} links recebidos")
except:
    pass


SEED: Cristiano Ronaldo
➤ [L0] Acessando: Cristiano Ronaldo... OK.
  ➤ [L1] Acessando: FIFA... OK.
  ➤ [L1] Acessando: UEFA... OK.
  ➤ [L1] Acessando: Portugal... OK.
  ➤ [L1] Acessando: Ballon d'Or... OK.
  ➤ [L1] Acessando: Juventus... OK.
  ➤ [L1] Acessando: Real Madrid... OK.
  ➤ [L1] Acessando: Madeira... OK.
  ➤ [L1] Acessando: Manchester United... OK.
  ➤ [L1] Acessando: Premier League... OK.
  ➤ [L1] Acessando: UEFA Champions League... OK.
  ➤ [L1] Acessando: Serie A... OK.
  ➤ [L1] Acessando: La Liga... OK.
  ➤ [L1] Acessando: Lionel Messi... OK.
  ➤ [L1] Acessando: European Golden Shoe... OK.
  ➤ [L1] Acessando: UEFA Nations League... OK.
  ➤ [L1] Acessando: Funchal... OK.
  ➤ [L1] Acessando: FIFA Club World Cup... OK.
  ➤ [L1] Acessando: Saudi Pro League... OK.
  ➤ [L1] Acessando: Instagram... OK.
  ➤ [L1] Acessando: Euro 2016... OK.
  ➤ [L1] Acessando: UEFA European Championship... OK.
  ➤ [L1] Acessando: Twitter... OK.
  ➤ [L1] Acessando: Forbes... OK.
  ➤ [L1] Acessando:

/home/eduardo/Documentos/dca3702-aed2/unity_3/venv/lib/python3.12/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /home/eduardo/Documentos/dca3702-aed2/unity_3/venv/lib/python3.12/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


OK.
    ➤ [L2] Acessando: European Championship... OK.
    ➤ [L2] Acessando: Stade de France... OK.
    ➤ [L2] Acessando: Nice... OK.
    ➤ [L2] Acessando: Toulouse... OK.
    ➤ [L2] Acessando: Saint-Étienne... OK.
    ➤ [L2] Acessando: Villeneuve-d'Ascq... OK.
    ➤ [L2] Acessando: Décines-Charpieu... OK.
    ➤ [L2] Acessando: Bordeaux... OK.
    ➤ [L2] Acessando: 2017 FIFA Confederations Cup... OK.
    ➤ [L2] Acessando: UEFA national team coefficient... OK.
    ➤ [L2] Acessando: Pro Evolution Soccer 2016... OK.
    ➤ [L2] Acessando: UEFA Euro 1984... OK.
    ➤ [L2] Acessando: Violence at UEFA Euro 2016... OK.
    ➤ [L2] Acessando: UEFA European Championship video games... OK.
    ➤ [L2] Acessando: UEFA European Championship top goalscorers... OK.
    ➤ [L2] Acessando: UEFA European Championship awards... OK.
    ➤ [L2] Acessando: UEFA Euro 2032 bids... OK.
    ➤ [L2] Acessando: UEFA Euro 2032... OK.
    ➤ [L2] Acessando: UEFA Euro 2028 qualifying... OK.
    ➤ [L2] Acessando: UEFA Eur